# Фаза по усреднённому циклу

## Импорты и параметры

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist
from my_lib import HankelMatrix, prepare_time_series

pio.renderers.default = "plotly_mimetype"
plt.rcParams.update({"figure.figsize": (10, 4), "font.family": "DejaVu Sans"})
np.random.seed(0)
WINDOW = 500
PERIOD = 460
N_CYCLES = 18
N_PHASE_POINTS = 4000

## Запись ходьбы

Период 460 отсчётов и границы отрезка заданы вручную.

In [2]:
x_acc, t = prepare_time_series('./data/long_walk_100_acc.csv')

dt = 460*20
x_acc = x_acc[7220:7220+dt]
fig = go.Figure()
fig.add_scatter(y = x_acc, mode='lines', name='x_acc')
fig.show()

## Окна, PCA и средний цикл

Усредняем 18 циклов по 460 точек.

In [3]:
X = HankelMatrix(x_acc, WINDOW)
pca = PCA(n_components=4, svd_solver="full")
X_PCA = pca.fit_transform(X)

cycles = X_PCA[:N_CYCLES * PERIOD].reshape(N_CYCLES, PERIOD, 4)
X_mean = cycles.mean(axis=0)
X_std = cycles.std(axis=0, ddof=1)
D = 2 * X_std.mean()
phi_grid = 2 * np.pi * np.arange(PERIOD) / PERIOD

In [4]:
fig_2 = go.Figure()
fig_2.update_layout(autosize=False, width=700, height=700)
fig_2.add_trace(go.Scatter3d(x=X_PCA[:460,0],
                             y=X_PCA[:460,1],
                             z=X_PCA[:460,2],
                             marker=dict(size=0.1,line=dict(width=0.01)),
                             name = 'Первый цикл'
                            )
                )
fig_2.add_trace(go.Scatter3d(x=X_mean[:,0],
                             y=X_mean[:,1],
                             z=X_mean[:,2],
                             marker=dict(size=0.1,line=dict(width=0.01)),
                             name = 'Средняя траектория'
                            )
                )
fig_2.layout.template = 'plotly_white'
fig_2.show()

## Кандидаты фаз

Считаем расстояния от окон до средней кривой.

In [5]:
distances_to_mean = cdist(X_PCA, X_mean)

## Выбор фазы

Для каждого кандидата считаем `L1 + L2`: штрафы за ход назад и расхождение с фазами соседей из истории.

In [6]:
phase_history = []
x_phase_history = {}
n_steps = 0
step_eps = 0.2 * np.pi
last_cycle_index = 0

for i, point in enumerate(X_PCA[:N_PHASE_POINTS]):
    if i == 0:
        phase = phi_grid[np.argmin(distances_to_mean[i])]
    else:
        candidate_indexes = np.flatnonzero(distances_to_mean[i] <= D)
        if len(candidate_indexes) == 0:
            raise ValueError(f"У окна {i} нет кандидатов в радиусе D={D:.3f}")
        candidate_phases = phi_grid[candidate_indexes] + 2 * np.pi * n_steps
        wrapped_candidates = candidate_phases % (2 * np.pi)

        previous_phase = phase_history[-1]
        loss1 = np.maximum(0, previous_phase - candidate_phases)
        neighbouring_phases = np.array([
            phase % (2 * np.pi)
            for phase, previous_point in x_phase_history.items()
            if np.linalg.norm(point - previous_point) <= D / 2
        ])
        loss2 = np.abs(neighbouring_phases[:, None] - wrapped_candidates[None, :]).sum(axis=0)
        phase = candidate_phases[np.argmin(loss1 + loss2)]

        if 2 * np.pi - phase % (2 * np.pi) <= step_eps:
            if i - last_cycle_index > 100:
                last_cycle_index = i
                n_steps += 1

    x_phase_history[phase] = point.copy()
    phase_history.append(phase)

phase_history = np.asarray(phase_history)

## Оценённая фаза

Показаны первые 4000 окон обучающего отрезка. `L3` в этом варианте не реализована.

In [7]:
fig = go.Figure()
fig.add_scatter(y=phase_history % (2 * np.pi), mode="lines", name="Фаза")
fig.update_layout(xaxis_title="Индекс окна", yaxis_title="Фаза, рад")
fig.show()